In [1]:
import os
import json
import numpy as np
import pandas as pd

import plotly.io as pio
import plotly.graph_objects as go
import plotly.express as px

from sklearn.cluster import DBSCAN
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
)
from sklearn.neighbors import NearestNeighbors
from sklearn.utils import resample

np.random.seed(42)
pio.templates.default = "simple_white"
_COLOR_PLOT = '#2f2f2f'

In [2]:
__author__ = ['Aleksandar Anžel']
__copyright__ = ''
__credits__ = ['Aleksandar Anžel', 'Zewen Yang', 'Georges Hattab']
__license__ = 'GNU General Public License v3.0'
__version__ = '1.0.0'
__maintainer__ = 'Aleksandar Anžel'
__email__ = 'AnzelA@rki.de'
__status__ = 'Stable'

In [3]:
path_dashboard_data_root = os.path.join('..', 'data')
path_climate = os.path.join(
    path_dashboard_data_root, 'Case_Study_Climate', 'climate_models_tmp.csv')
path_sample_analysis = os.path.join(
    path_dashboard_data_root, 'Case_Study_Wine', 'climate_models_tmp.csv')
path_dbscan_results = 'DBSCAN_Results'

os.makedirs(path_dbscan_results, exist_ok=True)

# 1. DBSCAN Sensitivity Analysis
Revision 1 requirement: Assess the influence of DBSCAN parameter selection on the clusters in the overview diagram of the proposed dashboard.

In [4]:
def auto_dbscan(X):
    n_dims = X.shape[1]
    n_samples = X.shape[0]

    k = max(2, min(2 * n_dims - 1, int(np.log(n_samples))))
    k = min(k, n_samples - 1)

    if k < 2:
        return 0.5, 2

    nbrs = NearestNeighbors(n_neighbors=k).fit(X)
    distances, _ = nbrs.kneighbors(X)
    k_distances = np.sort(distances[:, k - 1])

    if len(k_distances) < 3:
        return float(np.median(k_distances)), k

    diffs = np.diff(k_distances)
    second_diffs = np.diff(diffs)
    elbow_idx = np.argmax(second_diffs) + 1
    eps = k_distances[elbow_idx]

    min_pts = k
    return eps, min_pts

In [5]:
def prepare_dbscan_input(df_left_input, string_reference_model, list_measures):
    df_reference_row = df_left_input.loc[
        df_left_input["Model"] == string_reference_model
    ]

    df_input_no_reference = df_left_input.drop(df_reference_row.index)[
        list_measures
    ].copy()

    X = df_input_no_reference.values

    return {
        "df_reference_row": df_reference_row,
        "df_input_no_reference": df_input_no_reference,
        "X": X,
    }

In [7]:
def run_dbscan_with_auto_params(df_left_input, string_reference_model, list_measures):
    prepared = prepare_dbscan_input(
        df_left_input=df_left_input,
        string_reference_model=string_reference_model,
        list_measures=list_measures,
    )

    df_reference_row = prepared["df_reference_row"]
    df_input_no_reference = prepared["df_input_no_reference"]
    X = prepared["X"]

    float_eps, int_min_samples = auto_dbscan(X)

    constructor_DBSCAN = DBSCAN(
        eps=float_eps,
        min_samples=int_min_samples,
        n_jobs=-1,
    )
    constructor_DBSCAN.fit(X)

    labels_no_reference = list(constructor_DBSCAN.labels_)

    labels_with_reference = list(labels_no_reference)
    labels_with_reference.insert(
        df_reference_row.index.values[0],
        df_left_input.shape[0],
    )

    core_mask = np.zeros_like(constructor_DBSCAN.labels_, dtype=bool)
    if hasattr(constructor_DBSCAN, "core_sample_indices_"):
        core_mask[constructor_DBSCAN.core_sample_indices_] = True

    return {
        "eps": float_eps,
        "min_samples": int_min_samples,
        "df_reference_row": df_reference_row,
        "df_input_no_reference": df_input_no_reference,
        "X": X,
        "dbscan_model": constructor_DBSCAN,
        "labels_no_reference": labels_no_reference,
        "labels_with_reference": labels_with_reference,
        "core_mask": core_mask,
    }

In [8]:
def compute_validation_metrics(X, dbscan_model, core_mask):
    labels = dbscan_model.labels_
    noise_mask = labels == -1

    n_noise = int(np.sum(noise_mask))
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise_frac = n_noise / len(labels)

    n_core = int(np.sum(core_mask))
    n_border = int(np.sum((labels != -1) & (~core_mask)))

    valid_mask = labels != -1
    X_valid = X[valid_mask]
    labels_valid = labels[valid_mask]

    if len(set(labels_valid)) >= 2:
        silhouette = silhouette_score(X_valid, labels_valid)
        davies_bouldin = davies_bouldin_score(X_valid, labels_valid)
        calinski_harabasz = calinski_harabasz_score(X_valid, labels_valid)
    else:
        silhouette = np.nan
        davies_bouldin = np.nan
        calinski_harabasz = np.nan

    return {
        "n_clusters": n_clusters,
        "n_noise": n_noise,
        "noise_frac": noise_frac,
        "n_core": n_core,
        "n_border": n_border,
        "silhouette": silhouette,
        "davies_bouldin": davies_bouldin,
        "calinski_harabasz": calinski_harabasz,
    }

In [9]:
def sensitivity_sweep_for_grid_search(
    df_left_input,
    string_reference_model,
    list_measures,
    eps_center,
    min_samples_center,
    eps_range_ratio=0.30,
    eps_steps=41,
    min_samples_offsets=(-2, -1, 0, 1, 2),
):
    prepared = prepare_dbscan_input(
        df_left_input=df_left_input,
        string_reference_model=string_reference_model,
        list_measures=list_measures,
    )

    X = prepared["X"]

    eps_values = np.linspace(
        eps_center * (1 - eps_range_ratio),
        eps_center * (1 + eps_range_ratio),
        eps_steps,
    )

    rows = []

    for eps in eps_values:
        for offset in min_samples_offsets:
            min_samples = max(2, min_samples_center + offset)

            db = DBSCAN(
                eps=float(eps),
                min_samples=int(min_samples),
                n_jobs=-1,
            )
            db.fit(X)

            labels = db.labels_

            core_mask = np.zeros_like(labels, dtype=bool)
            if hasattr(db, "core_sample_indices_"):
                core_mask[db.core_sample_indices_] = True

            metrics = compute_validation_metrics(X, db, core_mask)

            rows.append(
                {
                    "eps": float(eps),
                    "min_samples": int(min_samples),
                    **metrics,
                }
            )

    return pd.DataFrame(rows)

In [10]:
def bootstrap_stability_for_grid_search(
    df_left_input,
    string_reference_model,
    list_measures,
    eps,
    min_samples,
    n_boot=100,
    sample_frac=0.8,
    random_state=42,
):
    prepared = prepare_dbscan_input(
        df_left_input=df_left_input,
        string_reference_model=string_reference_model,
        list_measures=list_measures,
    )

    X = prepared["X"]

    db_full = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1)
    db_full.fit(X)
    full_labels = db_full.labels_

    rng = np.random.RandomState(random_state)
    rows = []

    for i in range(n_boot):
        idx = resample(
            np.arange(X.shape[0]),
            replace=False,
            n_samples=max(3, int(X.shape[0] * sample_frac)),
            random_state=rng.randint(0, 10_000_000),
        )

        X_sub = X[idx]

        db_sub = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1)
        db_sub.fit(X_sub)
        sub_labels = db_sub.labels_

        full_labels_sub = full_labels[idx]
        ari_vs_full = adjusted_rand_score(full_labels_sub, sub_labels)

        rows.append(
            {
                "bootstrap_iter": i,
                "sample_size": len(idx),
                "ari_vs_full": float(ari_vs_full),
                "n_clusters": len(set(sub_labels)) - (1 if -1 in sub_labels else 0),
                "noise_frac": float(np.mean(sub_labels == -1)),
            }
        )

    return pd.DataFrame(rows)

In [11]:
def plot_k_distance_for_current_data(df_left_input, string_reference_model, list_measures, k, eps):
    prepared = prepare_dbscan_input(
        df_left_input=df_left_input,
        string_reference_model=string_reference_model,
        list_measures=list_measures,
    )
    X = prepared["X"]

    nbrs = NearestNeighbors(n_neighbors=k).fit(X)
    distances, _ = nbrs.kneighbors(X)
    k_distances = np.sort(distances[:, k - 1])

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=np.arange(len(k_distances)),
            y=k_distances,
            mode="lines",
            name="k-distance",
        )
    )
    fig.add_hline(
        y=eps,
        line_dash="dash",
        annotation_text=f"chosen eps = {eps:.3f}",
        annotation_position="top left",
    )
    fig.update_layout(title="k-distance elbow (DBSCAN)")
    fig.update_xaxes(title_text="Sorted pts")
    fig.update_yaxes(title_text="k-dist")
    return fig


def plot_silhouette_line(df_sweep, eps0, min_samples0):
    df_plot = df_sweep[df_sweep["min_samples"] == min_samples0].copy()

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=df_plot["eps"],
            y=df_plot["silhouette"],
            mode="lines+markers",
            name=f"min_samples={min_samples0}",
        )
    )
    fig.add_vline(
        x=eps0,
        line_dash="dash",
        annotation_text=f"chosen eps = {eps0:.3f}",
        annotation_position="top right",
    )
    fig.update_layout(
        title="Silhouette vs eps (fixed min_samples)",
        legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
    )
    fig.update_xaxes(title_text="eps")
    fig.update_yaxes(title_text="silhouette")
    fig.update_traces(cliponaxis=False)
    return fig


def plot_heatmap(df_sweep, value_col, title_text, colorbar_title):
    pivot = df_sweep.pivot_table(index="min_samples", columns="eps", values=value_col)
    pivot = pivot.sort_index()

    fig = go.Figure(
        data=go.Heatmap(
            z=pivot.values,
            x=np.round(pivot.columns.values, 4),
            y=pivot.index.values,
            colorbar=dict(title=colorbar_title),
        )
    )
    fig.update_layout(title=title_text)
    fig.update_xaxes(title_text="eps")
    fig.update_yaxes(title_text="min_samp")
    return fig


def plot_bootstrap_hist(df_boot):
    fig = go.Figure()
    fig.add_trace(
        go.Histogram(
            x=df_boot["ari_vs_full"],
            nbinsx=20,
            name="ARI",
        )
    )
    fig.update_layout(title="Bootstrap stability (ARI)")
    fig.update_xaxes(title_text="ARI")
    fig.update_yaxes(title_text="Count")
    return fig

In [12]:
run_result = run_dbscan_with_auto_params(
    df_left_input=df_left_input,
    string_reference_model=string_reference_model,
    list_measures=list_measures,
)

eps0 = run_result["eps"]
min_samples0 = run_result["min_samples"]
X = run_result["X"]

metrics0 = compute_validation_metrics(
    X=X,
    dbscan_model=run_result["dbscan_model"],
    core_mask=run_result["core_mask"],
)

df_sweep = sensitivity_sweep_for_grid_search(
    df_left_input=df_left_input,
    string_reference_model=string_reference_model,
    list_measures=list_measures,
    eps_center=eps0,
    min_samples_center=min_samples0,
    eps_range_ratio=0.30,
    eps_steps=41,
    min_samples_offsets=(-2, -1, 0, 1, 2),
)

df_boot = bootstrap_stability_for_grid_search(
    df_left_input=df_left_input,
    string_reference_model=string_reference_model,
    list_measures=list_measures,
    eps=eps0,
    min_samples=min_samples0,
    n_boot=100,
    sample_frac=0.8,
    random_state=42,
)

metrics0, df_sweep.head(), df_boot.head()

NameError: name 'df_left_input' is not defined

In [ ]:
chart_1 = plot_k_distance_for_current_data(
    df_left_input=df_left_input,
    string_reference_model=string_reference_model,
    list_measures=list_measures,
    k=min_samples0,
    eps=eps0,
)

chart_2 = plot_silhouette_line(df_sweep, eps0=eps0, min_samples0=min_samples0)
chart_3 = plot_heatmap(df_sweep, "n_clusters", "Clusters across eps/min_samples", "Clusters")
chart_4 = plot_heatmap(df_sweep, "noise_frac", "Noise fraction across eps/min_samples", "Noise")
chart_5 = plot_heatmap(df_sweep, "silhouette", "Silhouette across eps/min_samples", "Silhouette")
chart_6 = plot_bootstrap_hist(df_boot)

chart_1.show()
chart_2.show()
chart_3.show()
chart_4.show()
chart_5.show()
chart_6.show()

In [ ]:
df_sweep.to_csv(os.path.join(path_dbscan_results, "sensitivity_sweep.csv"), index=False)
df_boot.to_csv(os.path.join(path_dbscan_results, "bootstrap_stability.csv"), index=False)

chart_1.write_html(os.path.join(path_dbscan_results, "k_distance.html"))
chart_2.write_html(os.path.join(path_dbscan_results, "silhouette_vs_eps.html"))
chart_3.write_html(os.path.join(path_dbscan_results, "clusters_heatmap.html"))
chart_4.write_html(os.path.join(path_dbscan_results, "noise_heatmap.html"))
chart_5.write_html(os.path.join(path_dbscan_results, "silhouette_heatmap.html"))
chart_6.write_html(os.path.join(path_dbscan_results, "bootstrap_ari_hist.html"))

summary = {
    "eps0": float(eps0),
    "min_samples0": int(min_samples0),
    "baseline_metrics": {
        k: (float(v) if pd.notnull(v) else None) for k, v in metrics0.items()
    },
    "bootstrap_ari_mean": float(df_boot["ari_vs_full"].mean()),
    "bootstrap_ari_std": float(df_boot["ari_vs_full"].std()),
    "bootstrap_ari_median": float(df_boot["ari_vs_full"].median()),
}

with open(os.path.join(path_dbscan_results, "validation_summary.json", "w")) as f:
    json.dump(summary, f, indent=2)

summary